In [2]:
import os
import re
import time
import ee
import numpy as np
import pandas as pd

PROJECT_ID = "acoustic-gizmo-513018-a8"  # يمكن استبداله بمعرف مشروع زميلك عند الحاجة
ee.Initialize(project=PROJECT_ID)

HUBS = [
    ("Kafr_ElSheikh", 31.11, 30.94), ("Dakahlia", 31.04, 31.38),
    ("Sharqia", 30.72, 31.67), ("Beheira", 31.03, 30.47),
    ("Gharbia", 30.79, 31.00), ("Menoufia", 30.55, 31.01),
    ("Qalyubia", 30.46, 31.18), ("Salhia_Reclaimed", 30.65, 31.95),
    ("Fayoum", 29.31, 30.84), ("Beni_Suef", 29.07, 31.09),
    ("Minya", 28.09, 30.75), ("Assiut", 27.18, 31.18),
    ("Sohag", 26.56, 31.69), ("Qena_Luxor", 25.69, 32.64),
]

hubs_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([lon, lat]).buffer(3000), {"Governorate": name})
    for name, lat, lon in HUBS
])

BANDS = ["SMAP_Surface_Moisture_cm3", "SMAP_RootZone_Moisture_cm3", "SMAP_Profile_Moisture_cm3", "SMAP_Moisture_Anomaly"]
smap_col = ee.ImageCollection("NASA/SMAP/SPL4SMGP/008").select(
    ["sm_surface", "sm_rootzone", "sm_profile", "sm_surface_anomaly"], BANDS
)


def safe_mean(col, band_names):
    dummy = ee.Image.constant([0] * len(band_names)).toFloat().rename(band_names).updateMask(ee.Image.constant(0))
    return col.map(lambda img: img.toFloat()).merge(ee.ImageCollection([dummy])).mean()


def get_info_with_retry(ee_obj, retries=5):
    for attempt in range(retries):
        try:
            return ee_obj.getInfo()
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait_s = 15 * (attempt + 1)
            print(f"      ⏳ ضغط مؤقت على الخادم ({e}) -> انتظار {wait_s} ثانية...", flush=True)
            time.sleep(wait_s)


date_bins = pd.date_range("2016-01-01", "2025-12-31", freq="10D")
expected_dates = pd.to_datetime(date_bins[:-1])
intervals = [(date_bins[i].strftime("%Y-%m-%d"), date_bins[i + 1].strftime("%Y-%m-%d")) for i in range(len(date_bins) - 1)]

records_dict = {}
print("📥 [3/6] بدء سحب بيانات NASA SMAP L4 (تجميع سنوي موحد)...", flush=True)

for yr in range(2016, 2026):
    yr_int = [x for x in intervals if x[0].startswith(str(yr))]
    yr_stack = ee.ImageCollection([
        safe_mean(smap_col.filterDate(s, e), BANDS).rename([f"{s}_{b}" for b in BANDS])
        for s, e in yr_int
    ]).toBands()

    fc_yr = yr_stack.reduceRegions(collection=hubs_fc, reducer=ee.Reducer.mean(), scale=5000)
    res = get_info_with_retry(fc_yr)

    for feat in res["features"]:
        props = feat["properties"]
        gov = props["Governorate"]
        for k, val in props.items():
            m = re.match(r"^(?:\d+_)?(\d{4}-\d{2}-\d{2})_(.+)$", k)
            if m:
                records_dict.setdefault((gov, m.group(1)), {})[m.group(2)] = val

    print(f"   ✓ SMAP: اكتمل عام {yr}", flush=True)
    time.sleep(2.0)

df_raw = pd.DataFrame([{"Governorate": gov, "Date": dt, **vals} for (gov, dt), vals in records_dict.items()])
df_raw["Date"] = pd.to_datetime(df_raw["Date"])

grid = pd.MultiIndex.from_product([[h[0] for h in HUBS], expected_dates], names=["Governorate", "Date"]).to_frame().reset_index(drop=True)
out = pd.merge(grid, df_raw, on=["Governorate", "Date"], how="left").sort_values(["Governorate", "Date"]).reset_index(drop=True)

num_cols = out.select_dtypes(include=[np.number]).columns
out[num_cols] = out.groupby("Governorate")[num_cols].transform(lambda g: g.interpolate("linear").ffill().bfill())
out["Date"] = out["Date"].dt.strftime("%Y-%m-%d")

output_path = r"d:\vscode\NASA\DATA\03_SMAP_SoilMoisture.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
out.to_csv(output_path, index=False)
print(f"✅ اكتمل الملف الثالث: {output_path} | الأبعاد: {out.shape}\n")

📥 [3/6] بدء سحب بيانات NASA SMAP L4 (تجميع سنوي موحد)...
   ✓ SMAP: اكتمل عام 2016


C:\Users\eng mohamed\AppData\Roaming\Python\Python314\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


   ✓ SMAP: اكتمل عام 2017
   ✓ SMAP: اكتمل عام 2018
   ✓ SMAP: اكتمل عام 2019
   ✓ SMAP: اكتمل عام 2020
   ✓ SMAP: اكتمل عام 2021
   ✓ SMAP: اكتمل عام 2022
   ✓ SMAP: اكتمل عام 2023
   ✓ SMAP: اكتمل عام 2024
   ✓ SMAP: اكتمل عام 2025
✅ اكتمل الملف الثالث: d:\vscode\NASA\DATA\03_SMAP_SoilMoisture.csv | الأبعاد: (5110, 6)

